# 03 · Approach 1: 2-D CNN on log-mel spectrograms
Owner: **Member 3** · Runtime: Colab **GPU** (Runtime → Change runtime type → T4 GPU)

**Why this model.** A spoken word is a sequence of *local* time-frequency events (a burst, a
formant transition, a nasal). Stacked 3×3 convolutions learn these local patterns, pooling makes them
tolerant to small shifts in time and pitch, and depth grows the receptive field until it spans the
whole word. CNNs are the standard strong model for small-vocabulary keyword spotting
(Sainath & Parada, 2015; Warden, 2018). **Limitation:** order is only captured within the receptive
field, and there is no explicit memory of the whole sequence as in an RNN.

Experiments change **one thing at a time**, and each is motivated by what the previous run showed.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive to the (much faster) local Colab disk. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv`, `SampleSubmission.csv` and `Audio.zip`
(downloaded from Zindi) in a Google Drive folder called `swahili_audio`, and set `REPO_URL`.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-audio-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-audio-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_audio")
    LOCAL_DATA = Path("/content/data")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if not (LOCAL_DATA / ".ready").exists():
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        for f in DRIVE_DATA.glob("*.csv"):
            shutil.copy(f, LOCAL_DATA)
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_DATA / "Audio.zip"), "-d", str(LOCAL_DATA)], check=True)
        (LOCAL_DATA / ".ready").touch()
    os.environ["DATA_DIR"] = str(LOCAL_DATA)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")          # features persist between sessions
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")  # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import torch

from src import config as C
from src.data import get_splits, label_encoder, encode, load_unlabelled, add_paths
from src.datasets import ArrayDataset, SpecAugment
from src.evaluate import show_log
from src.features import compute_features, Standardizer
from src.models import LogMelCNN
from src.pipeline import run_experiments, finalize
from src.utils import set_seed, get_device

MEMBER, MODEL = "Member 3", "CNN log-mel"
set_seed(C.SEED)
print("Device:", get_device())
df = get_splits()
classes, to_id = label_encoder(df.label)
K = len(classes)
tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
ytr, yva, yte = (encode(d.label, to_id) for d in (tr, va, te))
zindi = add_paths(load_unlabelled())

## Features: log-mel spectrograms (64 mel bands × ~100 frames per second)

In [ ]:
_feat_cache = {}


def features(trim=True):
    """Standardised log-mel arrays for train / val / test / zindi (cached)."""
    if trim not in _feat_cache:
        X = [compute_features(d.path, "logmel", trim=trim) for d in (tr, va, te, zindi)]
        sc = Standardizer().fit(X[0])                     # statistics from TRAIN only
        _feat_cache[trim] = [sc.transform(x) for x in X]
    return _feat_cache[trim]


print("log-mel input shape (n_mels, frames):", features()[0].shape[1:])


def make_datasets(p):
    Xtr, Xva, _, _ = features(p.get("trim", True))
    aug = SpecAugment() if p.get("augment") else None
    return ArrayDataset(Xtr, ytr, aug), ArrayDataset(Xva, yva)


def build_model(p):
    return LogMelCNN(K, channels=p.get("channels", (32, 64, 128, 256)), dropout=p.get("dropout", 0.2))

## Experiments

In [ ]:
DEFAULT_TRAIN = dict(epochs=40, lr=1e-3, batch_size=64, weight_decay=1e-4, patience=8)
EXPERIMENTS = [
    dict(id="CNN-01", description="4-block CNN, no augmentation",
         hypothesis="Local time-frequency patterns are enough to recognise isolated words.",
         motivated_by="Starting point", params={}),
    dict(id="CNN-02", description="+ SpecAugment and random time shift",
         hypothesis="Masking and shifting reduce overfitting to speaker- and position-specific details.",
         motivated_by="CNN-01 train/val gap + EDA: large within-class variability and shifted onsets",
         params={"augment": True}),
    dict(id="CNN-03", description="+ SpecAugment + label smoothing 0.1",
         hypothesis="Label smoothing reduces over-confident errors, which log loss punishes heavily.",
         motivated_by="CNN-02 reliability plot / confidently wrong predictions",
         params={"augment": True}, train={"label_smoothing": 0.1}),
    dict(id="CNN-04", description="Smaller CNN (16-32-64-128 channels) + SpecAugment",
         hypothesis="With a few thousand clips, a 4x smaller model generalises as well at lower cost.",
         motivated_by="Capacity check: is the dataset too small for the default width?",
         params={"augment": True, "channels": (16, 32, 64, 128)}),
    dict(id="CNN-05", description="Best setup but WITHOUT silence trimming",
         hypothesis="Without trimming, part of the input budget is wasted on silence and the word position varies more.",
         motivated_by="EDA section 4: silence is a large, variable share of each clip",
         params={"augment": True, "trim": False}),
]
best, summary = run_experiments(EXPERIMENTS, build_model, make_datasets, member=MEMBER, model_name=MODEL,
                                n_classes=K, y_train=ytr, default_train=DEFAULT_TRAIN)
summary.round(4)

> **Read the table before moving on:** which change helped, which did not, and why? Write it in the
> `observation` column of `experiments/log_cnn_log_mel.csv`. If an idea emerges (e.g. more dropout
> because the gap is still large), add a `CNN-06` entry above and re-run: that is the progression the
> rubric asks for.

## Final evaluation of the selected configuration on the test split (once)

In [ ]:
p = best["exp"]["params"]
_, _, Xte, Xz = features(p.get("trim", True))
model, proba_test, metrics = finalize(best, build_model, ArrayDataset(Xte, yte), yte, te.file.values, classes,
                                      key="cnn_logmel", display_name="CNN (log-mel)",
                                      zindi_ds=ArrayDataset(Xz), zindi_df=zindi)

In [ ]:
from src.utils import persist_outputs
persist_outputs()